### DESeq2 multifactor design test

In [1]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion, Formula
import rpy2.robjects.packages as rpackages
from rpy2.robjects.packages import importr

In [3]:
import os
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
from deconveil.utils_fit import *
from deconveil.utils_processing import *

#### Load TCGA data

In [7]:
rna_counts = load_test_data(
    modality="rna",
    dataset="tcga_brca",
    debug=False,
)
rna_counts = rna_counts.T

metadata = load_test_data(
    modality="metadata",
    dataset="tcga_brca",
    debug=False,
)

cnv = load_test_data(
    modality="cnv",
    dataset="tcga_brca",
    debug=False,
)
cnv = cnv.T

In [9]:
rna_counts.head()

,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2MP1,A3GALT2,A4GALT,A4GNT,AACS,AACSP1,...,ZSWIM6,ZSWIM9,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
GTEX-13PVQ-1026-SM-5KM3M.1,3502210.0,83739.0,1592.0,17369.0,9842.0,4961.0,377148.0,657.0,156623.0,151.0,...,231243.0,11494.0,41101.0,23345.0,4853.0,93143.0,679.0,97130.0,1819661.0,394464.0
GTEX-18QFQ-0826-SM-718AX.1,5394960.0,155914.0,14511.0,10089.0,11023.0,303.0,278056.0,357.0,385961.0,149.0,...,184259.0,83650.0,123117.0,74837.0,25442.0,320704.0,8603.0,300670.0,1580226.0,918922.0
GTEX-1JN6P-2426-SM-ARL99.1,7136304.0,219925.0,2254.0,12267.0,5955.0,769.0,229205.0,0.0,40360.0,0.0,...,114710.0,22251.0,37669.0,17233.0,4141.0,82340.0,372.0,89483.0,982597.0,339959.0
GTEX-13S86-1226-SM-5S2OA.1,5710693.0,236061.0,932.0,4446.0,12644.0,986.0,468005.0,76.0,177527.0,0.0,...,62143.0,22458.0,73701.0,20962.0,8415.0,111953.0,126.0,94875.0,1060350.0,233669.0
GTEX-132NY-0826-SM-5K7Y7.1,4020951.0,155161.0,6312.0,5744.0,5015.0,76.0,480082.0,1286.0,277306.0,152.0,...,106252.0,28196.0,49946.0,35748.0,6204.0,134309.0,505.0,133562.0,1136263.0,392324.0


In [11]:
metadata.head()

,condition
GTEX-13PVQ-1026-SM-5KM3M.1,A
GTEX-18QFQ-0826-SM-718AX.1,A
GTEX-1JN6P-2426-SM-ARL99.1,A
GTEX-13S86-1226-SM-5S2OA.1,A
GTEX-132NY-0826-SM-5K7Y7.1,A


In [13]:
cnv.head()

,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2MP1,A3GALT2,A4GALT,A4GNT,AACS,AACSP1,...,ZSWIM6,ZSWIM9,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
GTEX-13PVQ-1026-SM-5KM3M.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-18QFQ-0826-SM-718AX.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-1JN6P-2426-SM-ARL99.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-13S86-1226-SM-5S2OA.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
GTEX-132NY-0826-SM-5K7Y7.1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2


#### QC check and filtering

In [15]:
print("Before filtering:", rna_counts.shape, cnv.shape, metadata.shape)
# Reorder cnv to match rna_counts
cnv = cnv.loc[rna_counts.index]
assert (rna_counts.index == metadata.index).all(), "Sample order mismatch between rna_counts and metadata"
assert (cnv.index == rna_counts.index).all(), "Sample order mismatch between cnv and rna_counts"

Before filtering: (400, 17387) (400, 17387) (400, 1)


In [17]:
all_zero_mask = (rna_counts.sum(axis=0) == 0)
print("All-zero genes:", all_zero_mask.sum())
rna_counts = rna_counts.loc[:, ~all_zero_mask]
cnv = cnv.loc[:, ~all_zero_mask]  

All-zero genes: 0


In [19]:
res = filter_low_count_genes(rna_counts, other_dfs=[cnv], min_count=300, min_samples=50)
rna_counts = res["filtered_df"]
cnv = res["other_filtered"][0]
print("After low-count filtering:", rna_counts.shape, cnv.shape)

After low-count filtering: (400, 16252) (400, 16252)


In [ ]:
def run_deseq2_multifactor(rna_counts, 
                           metadata, 
                           cnv_matrix, 
                           design_formula="~condition * CN",
                           shrink_coef=None,
                           shrink_type="apeglm"):
    """
    Run DESeq2 multifactor model integrating CN (copy-number) information as a continuous covariate.
    Automatically computes mean CN per sample from a genes x samples CNV matrix.

    Parameters
    ----------
    rna_counts : pd.DataFrame
        Gene expression count matrix (genes x samples)
    metadata : pd.DataFrame
        Sample-level metadata (samples x covariates)
    cnv_matrix : pd.DataFrame
        CNV data matrix (genes x samples)
    design_formula : str
        R-style formula for DESeq2 (e.g. "~ condition + CN" or "~ condition * CN")
    shrink_coef : str, optional
        Name of coefficient for LFC shrinkage (must match resultsNames(dds))
    shrink_type : str, optional
        Shrinkage estimator type (default: "apeglm")

    Returns
    -------
    res_df : pd.DataFrame
        DESeq2 results table
    merged_metadata : pd.DataFrame
        Metadata with CN column added
    """

    deseq2 = importr("DESeq2")

    # Align samples 
    shared_samples = (
        rna_counts.columns
        .intersection(metadata.index)
        .intersection(cnv_matrix.columns)
    )

    if len(shared_samples) == 0:
        raise ValueError("No overlapping samples found across inputs.")

    rna_counts = rna_counts[shared_samples]
    metadata = metadata.loc[shared_samples]
    cnv_matrix = cnv_matrix[shared_samples]

    if rna_counts.shape[0] < rna_counts.shape[1]:
        print("Transposing RNA count matrix to match DESeq2 format (genes as rows).")
        rna_counts = rna_counts.T

    # Compute mean CN per sample 
    metadata["CNmean"] = cnv_matrix.mean(axis=0).astype(float)

    # Assign CN status
    metadata["CNstatus"] = pd.cut(
        metadata["CNmean"],
        bins=[-np.inf, 1.8, 2.2, np.inf],
        labels=["Loss", "Neutral", "Gain"]
    )

    # Convert all object columns to categorical, except CNmean
    cat_cols = [
        c for c in metadata.select_dtypes(include=["object", "category"]).columns
        if c != "CNmean"
    ]
    
    for col in cat_cols:
        metadata[col] = metadata[col].astype("category")

    # Ensure CNmean stats numeric
    if "CNmean" in metadata.columns:
        metadata["CNmean"] = pd.to_numeric(metadata["CNmean"], errors="coerce")

    # Index and type preparation 
    rna_counts.index = rna_counts.index.astype(str)
    metadata.index = metadata.index.astype(str)
   
    # Convert to R objects 
    with conversion.localconverter(ro.default_converter + pandas2ri.converter):
        rna_counts_r = conversion.py2rpy(rna_counts.astype(int))
        metadata_r = conversion.py2rpy(metadata)

    # Assign to R environment 
    ro.globalenv["rna_counts_r"] = rna_counts_r
    ro.globalenv["metadata_r"] = metadata_r
    ro.globalenv["gene_names"] = ro.StrVector(rna_counts.index.tolist())
    ro.globalenv["sample_names"] = ro.StrVector(metadata.index.tolist())
    ro.r("rownames(rna_counts_r) <- gene_names")
    ro.r("rownames(metadata_r) <- sample_names")

    # ensure all factors are unordered in R
    ro.r('''
    for (col in colnames(metadata_r)) {
      if (is.factor(metadata_r[[col]]) && is.ordered(metadata_r[[col]])) {
        metadata_r[[col]] <- factor(metadata_r[[col]], ordered = FALSE)
      }
    }
    ''')

    # Validate sample alignment in R 
    alignment_check = ro.r('all(colnames(rna_counts_r) == rownames(metadata_r))')[0]
    if not alignment_check:
        raise ValueError("Sample names in rna_counts and metadata do not align after conversion.")

    # Run DESeq2 
    print("Running DESeq2 with design:", design_formula)
    dds = deseq2.DESeqDataSetFromMatrix(
        countData=ro.globalenv["rna_counts_r"],
        colData=ro.globalenv["metadata_r"],
        design=Formula(design_formula)
    )

    dds = deseq2.DESeq(dds)
    ro.globalenv["dds"] = dds

    # Show available coefficients ----
    coef_names = list(ro.r("resultsNames(dds)"))
    print("\nAvailable model coefficients:")
    for c in coef_names:
        print("  -", c)

    coef_names = list(ro.r("resultsNames(dds)"))
    if len(coef_names) == 0:
    # Fallback: use coef(dds) column names
        alt_coef_names = list(ro.r("colnames(coef(dds))"))
        print("\n resultsNames(dds) is empty — falling back to coef() column names:")
        print("  ", alt_coef_names)
        coef_names = alt_coef_names

    # Compute results (with optional shrinkage) 
    if shrink_coef is not None:
        if shrink_coef not in coef_names:
            raise ValueError(
                f"Requested shrink_coef '{shrink_coef}' not found in resultsNames(dds).\n"
                f"Available: {coef_names}"
            )
        print(f"\nPerforming LFC shrinkage for coefficient: {shrink_coef}")
        res_shr = deseq2.lfcShrink(
            dds,
            coef=shrink_coef,
            type=shrink_type
        )
        res_df_r = ro.r("as.data.frame")(res_shr)
    else:
        print("\nExtracting standard DESeq2 results (no shrinkage).")
        res = deseq2.results(dds)
        res_df_r = ro.r("as.data.frame")(res)

    # Convert back to pandas
    with conversion.localconverter(ro.default_converter + pandas2ri.converter):
        res_df = conversion.rpy2py(res_df_r)

    return dds, res_df 

#### Model test

In [21]:
res_df, dds = run_deseq2_multifactor(
    rna_counts=rna_counts.T,
    metadata=metadata,
    cnv_matrix=cnv.T,
    design_formula="~condition",
    shrink_coef="condition_B_vs_A",
    shrink_type="apeglm"
)

Running DESeq2 with design: ~condition


R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 1590 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  



Available model coefficients:
  - Intercept
  - condition_B_vs_A

Performing LFC shrinkage for coefficient: condition_B_vs_A


R callback write-console: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
  


#### Parse pandas dataframes in R format

In [91]:
rna_counts.index = rna_counts.index.astype(str)    # gene names
metadata.index   = metadata.index.astype(str)      # sample names

gene_names   = rna_counts.index.tolist()
sample_names = metadata.index.tolist()

In [93]:
with conversion.localconverter(ro.default_converter + pandas2ri.converter):
    rna_counts_r = conversion.py2rpy(rna_counts.astype(int))
    metadata_r   = conversion.py2rpy(metadata)

In [95]:
ro.globalenv['rna_counts_r'] = rna_counts_r
ro.globalenv['gene_names']   = ro.StrVector(gene_names)
ro.r('rownames(rna_counts_r) <- gene_names')

ro.globalenv['metadata_r'] = metadata_r
ro.globalenv['sample_names'] = ro.StrVector(sample_names)
ro.r('rownames(metadata_r) <- sample_names')

In [97]:
rna_counts_r = ro.globalenv['rna_counts_r']
metadata_r   = ro.globalenv['metadata_r']

In [99]:
print(ro.r('head(rownames(rna_counts_r))'))   # should show gene IDs
print(ro.r('head(rownames(metadata_r))'))     # should show sample names

[1] "A2M"       "A2M-AS1"   "A2ML1"     "A2ML1-AS1" "A2MP1"     "A3GALT2"  

[1] "GTEX-13PVQ-1026-SM-5KM3M.1" "GTEX-18QFQ-0826-SM-718AX.1"
[3] "GTEX-1JN6P-2426-SM-ARL99.1" "GTEX-13S86-1226-SM-5S2OA.1"
[5] "GTEX-132NY-0826-SM-5K7Y7.1" "GTEX-ZTX8-1226-SM-4YCE9.1" 



#### Create DESeq2 dataset | Model fit

In [101]:
deseq2 = importr("DESeq2")

In [103]:
print(ro.r('dim(rna_counts_r)'))
print(ro.r('dim(metadata_r)'))

[1] 16252   400

[1] 400   3



In [101]:
dds = deseq2.DESeqDataSetFromMatrix(
    countData=ro.globalenv["rna_counts_r"],
    colData=ro.globalenv["metadata_r"],
    design=ro.Formula("~ condition")  
)

In [103]:
dds = deseq2.DESeq(dds)

R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 1590 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  


In [105]:
ro.globalenv["dds"] = dds

In [107]:
print(ro.r("resultsNames(dds)"))

[1] "Intercept"        "condition_B_vs_A"



In [67]:
res_shr = deseq2.lfcShrink(
    dds,
    coef=ro.StrVector(["conditionB.CNNeutral"]),  
    type=ro.StrVector(["apeglm"])
)

R callback write-console: using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895
  


In [69]:
with conversion.localconverter(ro.default_converter + pandas2ri.converter):
    res_shr_df = conversion.rpy2py(ro.r('as.data.frame')(res_shr))

In [57]:
res_shr_df.head() # interaction - conditionB.CNLoss

,baseMean,log2FoldChange,lfcSE,pvalue,padj
A2M,5.180438e+06,0.016200,0.053867,0.172547,0.735072
A2M-AS1,1.578353e+05,0.031283,0.065030,0.012620,0.323401
A2ML1,6.184772e+04,0.013945,0.054217,0.005609,0.228917
A2ML1-AS1,7.217111e+03,-0.011413,0.052078,0.319978,0.826664
A2MP1,7.993756e+03,1.536313,0.225346,0.224821,0.775731


In [71]:
res_shr_df.head() # interaction - conditionB.CNNeutral

,baseMean,log2FoldChange,lfcSE,pvalue,padj
A2M,5.180438e+06,0.096873,0.163281,0.023639,0.273135
A2M-AS1,1.578353e+05,0.518660,0.305058,0.002455,0.086396
A2ML1,6.184772e+04,0.024720,0.089602,0.068287,0.441120
A2ML1-AS1,7.217111e+03,0.039598,0.091788,0.191504,0.642614
A2MP1,7.993756e+03,0.022127,0.083350,0.445340,0.832682
